
# gffcompare details

This notebook loads per-direction class_code counts from `qc_metrics/<assembly_accession>/gffcompare/class_counts_<direction>.tsv` and produces summary plots.

- Directions are treated independently: `Ensembl_to_CAT` and `CAT_to_Ensembl` have distinct denominators (query transcripts).
- Figures are saved under `results/figures/`.


In [15]:

import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

QC_DIR = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results/qc_metrics')  # adjust if needed
FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Discover class_counts files
files = list(QC_DIR.glob('*/gffcompare/class_counts_*.tsv'))
print(f"Found {len(files)} class_count files")

cols = [
    'assembly_accession','sample_name','direction','class_code','n_transcripts','denominator','pct'
]

dfs = []
for f in files:
    try:
        df = pd.read_csv(f, sep='	')
        # Validate columns and coerce types
        missing = [c for c in cols if c not in df.columns]
        if missing:
            print(f"Skipping {f}: missing columns {missing}")
            continue
        df = df[cols].copy()
        df['n_transcripts'] = pd.to_numeric(df['n_transcripts'], errors='coerce').fillna(0).astype(int)
        df['denominator'] = pd.to_numeric(df['denominator'], errors='coerce').fillna(0).astype(int)
        df['pct'] = pd.to_numeric(df['pct'], errors='coerce').fillna(0.0)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {f}: {e}")

if not dfs:
    print("No valid class_counts TSVs found; aborting plots.")
    raise SystemExit(0)

cc = pd.concat(dfs, ignore_index=True)

# Ensure directions are the exact expected labels
valid_dirs = ['Ensembl_to_CAT', 'CAT_to_Ensembl']
cc = cc[cc['direction'].isin(valid_dirs)].copy()

# Plot 1: Bar plots of median pct per class_code, one per direction
for d in valid_dirs:
    sub = cc[cc['direction']==d].copy()
    if sub.empty:
        print(f"No rows for {d}")
        continue
    med = (sub.groupby('class_code', as_index=False)['pct']
               .median()
               .sort_values('class_code'))
    plt.figure(figsize=(10,5))
    sns.barplot(data=med, x='class_code', y='pct', color='steelblue')
    plt.title(f"gffcompare class codes — {d} (Median % of query transcripts)")
    plt.ylabel('% of query transcripts')
    plt.xlabel('class_code')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_median_pct_{d}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"Saved {out}")

# Plot 2: Boxplots of denominators per direction
plt.figure(figsize=(8,5))
order = valid_dirs
sns.boxplot(data=cc.drop_duplicates(['assembly_accession','sample_name','direction','denominator']),
            x='direction', y='denominator', order=order)
plt.title('gffcompare denominators (query transcripts) by direction')
plt.ylabel('Number of query transcripts (denominator)')
plt.xlabel('Direction')
plt.tight_layout()
out = FIG_DIR / 'gffcompare_denominator_boxplots.png'
plt.savefig(out, dpi=150)
plt.close()
print(f"Saved {out}")

# Plot 3 (optional): stacked bar for a sample of assemblies per direction
# Take up to 12 assemblies for legibility
sample_acc = (cc.groupby('assembly_accession')['denominator']
                .sum()
                .sort_values(ascending=False)
                .head(12)
                .index.tolist())
stack = cc[cc['assembly_accession'].isin(sample_acc)].copy()
for d in valid_dirs:
    sub = stack[stack['direction']==d].copy()
    if sub.empty:
        continue
    pivot = sub.pivot_table(index='assembly_accession', columns='class_code', values='pct', aggfunc='sum').fillna(0)
    pivot = pivot[pivot.columns.sort_values()]
    ax = pivot.plot(kind='bar', stacked=True, figsize=(12,6), colormap='tab20')
    ax.set_title(f'Stacked % by class_code — {d} (Percent of query transcripts)')
    ax.set_ylabel('% of query transcripts')
    ax.set_xlabel('assembly_accession')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_stacked_pct_{d}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"Saved {out}")


Found 924 class_count files
Saved results/figures/gffcompare_median_pct_Ensembl_to_CAT.png
Saved results/figures/gffcompare_median_pct_CAT_to_Ensembl.png
Saved results/figures/gffcompare_denominator_boxplots.png
Saved results/figures/gffcompare_stacked_pct_Ensembl_to_CAT.png
Saved results/figures/gffcompare_stacked_pct_CAT_to_Ensembl.png


In [16]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

QC_DIR = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results/qc_metrics')  # adjust if needed
FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Discover class_counts files
files = list(QC_DIR.glob('*/gffcompare/class_counts_*.tsv'))
print(f"Found {len(files)} class_count files")

cols = [
    'assembly_accession', 'sample_name', 'direction',
    'class_code', 'n_transcripts', 'denominator', 'pct'
]

# Map gffcompare class codes to readable labels
CLASS_CODE_MAP = {
    '=': 'Exact intron-chain match',
    'c': 'Contained in reference',
    'k': 'Reference contained in query',
    'j': 'Novel isoform (junction match)',
    'e': 'Single exon overlap',
    'o': 'Generic exon overlap',
    'i': 'Fully within intron',
    'n': 'Intronic overlap',
    'm': 'Retained intron',
    'u': 'Intergenic',
    'x': 'Opposite strand overlap',
    's': 'Intron match opposite strand',
    'y': 'Reference contains query',
    'p': 'Possible polymerase run-on',
    'r': 'Repeat region',
    '.': 'Unclassified'
}

# Biologically sensible plotting order
CLASS_ORDER = [
    'Exact intron-chain match',
    'Contained in reference',
    'Reference contained in query',
    'Novel isoform (junction match)',
    'Retained intron',
    'Single exon overlap',
    'Generic exon overlap',
    'Intronic overlap',
    'Fully within intron',
    'Intergenic',
    'Opposite strand overlap',
    'Intron match opposite strand',
    'Reference contains query',
    'Repeat region',
    'Possible polymerase run-on',
    'Unclassified'
]

dfs = []
for f in files:
    try:
        df = pd.read_csv(f, sep='\t')
        missing = [c for c in cols if c not in df.columns]
        if missing:
            print(f"Skipping {f}: missing columns {missing}")
            continue
        df = df[cols].copy()
        df['n_transcripts'] = pd.to_numeric(df['n_transcripts'], errors='coerce').fillna(0).astype(int)
        df['denominator'] = pd.to_numeric(df['denominator'], errors='coerce').fillna(0).astype(int)
        df['pct'] = pd.to_numeric(df['pct'], errors='coerce').fillna(0.0)
        df['class_label'] = df['class_code'].map(CLASS_CODE_MAP).fillna(df['class_code'])
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {f}: {e}")

if not dfs:
    print("No valid class_counts TSVs found; aborting plots.")
    raise SystemExit(0)

cc = pd.concat(dfs, ignore_index=True)

# Ensure directions are the exact expected labels
valid_dirs = ['Ensembl_to_CAT', 'CAT_to_Ensembl']
cc = cc[cc['direction'].isin(valid_dirs)].copy()

if cc.empty:
    print("No rows remain after filtering to valid directions.")
    raise SystemExit(0)

# Plot 1: Bar plots of median pct per class, one per direction
for d in valid_dirs:
    sub = cc[cc['direction'] == d].copy()
    if sub.empty:
        print(f"No rows for {d}")
        continue

    med = (
        sub.groupby('class_label', as_index=False)['pct']
           .median()
    )
    med['class_label'] = pd.Categorical(med['class_label'], categories=CLASS_ORDER, ordered=True)
    med = med.sort_values('class_label')

    plt.figure(figsize=(12, 5.5))
    sns.barplot(data=med, x='class_label', y='pct', color='steelblue')
    plt.title(f"gffcompare classes — {d} (Median % of query transcripts)")
    plt.ylabel('% of query transcripts')
    plt.xlabel('Classification')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_median_pct_{d}_labels.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"Saved {out}")

# Plot 2: Boxplots of denominators per direction
denom_df = cc.drop_duplicates(['assembly_accession', 'sample_name', 'direction', 'denominator']).copy()
if not denom_df.empty:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=denom_df, x='direction', y='denominator', order=valid_dirs)
    plt.title('gffcompare denominators (query transcripts) by direction')
    plt.ylabel('Number of query transcripts (denominator)')
    plt.xlabel('Direction')
    plt.tight_layout()
    out = FIG_DIR / 'gffcompare_denominator_boxplots.png'
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"Saved {out}")

# Plot 3: stacked bar for a sample of assemblies per direction
sample_acc = (
    cc.groupby('assembly_accession')['denominator']
      .sum()
      .sort_values(ascending=False)
      .head(12)
      .index
      .tolist()
)

stack = cc[cc['assembly_accession'].isin(sample_acc)].copy()

for d in valid_dirs:
    sub = stack[stack['direction'] == d].copy()
    if sub.empty:
        continue

    pivot = (
        sub.pivot_table(
            index='assembly_accession',
            columns='class_label',
            values='pct',
            aggfunc='sum',
            fill_value=0
        )
        .fillna(0)
    )

    cols_present = [c for c in CLASS_ORDER if c in pivot.columns]
    other_cols = [c for c in pivot.columns if c not in cols_present]
    pivot = pivot[cols_present + sorted(other_cols)]

    ax = pivot.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
    ax.set_title(f'Stacked % by class — {d} (sample of assemblies)')
    ax.set_ylabel('% of query transcripts')
    ax.set_xlabel('assembly_accession')
    plt.xticks(rotation=45, ha='right')
    ax.legend(title='Classification', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_stacked_pct_{d}_labels.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {out}")

# Plot 4: stacked bar for ALL assemblies, sorted by exact match, one plot per direction
for d in valid_dirs:
    sub = cc[cc['direction'] == d].copy()
    if sub.empty:
        print(f"No rows for {d}")
        continue

    pivot = (
        sub.pivot_table(
            index='assembly_accession',
            columns='class_label',
            values='pct',
            aggfunc='sum',
            fill_value=0
        )
        .fillna(0)
    )

    # Ensure exact-match column exists so sorting does not fail
    exact_col = 'Exact intron-chain match'
    if exact_col not in pivot.columns:
        pivot[exact_col] = 0.0

    # Order columns biologically, then append any unexpected labels
    cols_present = [c for c in CLASS_ORDER if c in pivot.columns]
    other_cols = [c for c in pivot.columns if c not in cols_present]
    pivot = pivot[cols_present + sorted(other_cols)]

    # Sort assemblies by % exact match
    pivot = pivot.sort_values(exact_col, ascending=False)

    # Width scales with number of assemblies
    fig_w = max(12, min(30, len(pivot) * 0.08))

    ax = pivot.plot(
        kind='bar',
        stacked=True,
        figsize=(fig_w, 5.5),
        width=1.0,
        colormap='tab20'
    )

    ax.set_title(f'gffcompare class composition across all assemblies — {d}\nSorted by % exact intron-chain match')
    ax.set_ylabel('% of query transcripts')
    ax.set_xlabel('Assemblies (sorted by exact match)')
    ax.set_xticks([])
    ax.tick_params(axis='x', length=0)
    ax.legend(title='Classification', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

    plt.tight_layout()
    out = FIG_DIR / f"gffcompare_stacked_all_assemblies_sorted_by_exact_{d}_labels.png"
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"Saved {out}")

Found 924 class_count files
Saved results/figures/gffcompare_median_pct_Ensembl_to_CAT_labels.png
Saved results/figures/gffcompare_median_pct_CAT_to_Ensembl_labels.png
Saved results/figures/gffcompare_denominator_boxplots.png
Saved results/figures/gffcompare_stacked_pct_Ensembl_to_CAT_labels.png
Saved results/figures/gffcompare_stacked_pct_CAT_to_Ensembl_labels.png
Saved results/figures/gffcompare_stacked_all_assemblies_sorted_by_exact_Ensembl_to_CAT_labels.png
Saved results/figures/gffcompare_stacked_all_assemblies_sorted_by_exact_CAT_to_Ensembl_labels.png


In [17]:
cc[(cc['pct'] == 100) & (cc['class_code'] == 'u')]


,assembly_accession,sample_name,direction,class_code,n_transcripts,denominator,pct,class_label
